> **Gulf Council / FEI Data Pipeline — YouTube & Podcast (full version).** Two
> parts that transcribe spoken-word media into the shared `Data_Repository`:
> **Part 1** collects YouTube video captions; **Part 2** transcribes podcast
> episodes. Both keep only Gulf-policy content from the **last 10 years**.

# Part 1 — YouTube Transcript Scraper

Enumerates videos from Gulf-policy YouTube channels and keyword searches, fetches
their captions, and saves only transcripts that are about Gulf of Mexico fishery
policy and were uploaded in the **last 10 years** — as text + metadata under
`Data_Repository`.

## Setup — Install & Imports

Installs the YouTube tooling (yt-dlp, youtube-transcript-api) and imports the standard-library modules used throughout Part 1.

In [ ]:
!pip install pymupdf beautifulsoup4 duckduckgo-search curl_cffi yt-dlp youtube-transcript-api

from __future__ import annotations

import argparse
import csv
import json
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

## Policy & Gulf Relevance Filter

The strong/weak **policy** and **Gulf** keyword sets plus the matchers (`policy_hits`, `gulf_hits`) that decide whether a transcript is about Gulf fishery policy.

In [ ]:

POLICY_KEYWORDS: tuple[str, ...] = (
    "gulf-council", "gulf council", "gmfmc", "nmfs", "noaa", "fwc",
    "tpwd", "ldwf", "lwf", "wlf", "sefsc", "magnuson", "msa",
    "amendment", "regulation", "regulations", "rulemaking",
    "ifq", "quota", "allocation", "fmp", "mrip", "mrfss",
    "closure", "reopen", "open-season", "closed-season",
    "season-closure", "season-opening",
    "bag-limit", "size-limit", "slot-limit", "trip-limit", "vessel-limit",
    "stock-assessment", "overfishing", "overfished",
    "for-hire",
    "by-catch", "bycatch", "release-mortality",
    "descending-device", "descender", "venting-tool", "barotrauma",
    "enforcement", "poaching",
    "public-comment", "comment-period", "public-hearing", "public-meeting",
    "testimony",
    "ecosystem-based-management", "ebm", "ecosystem-assessment",
    "fishery-management", "fisheries-management",
    "sustainable-fishery", "sustainable-fisheries",
    "red-tide",
)

EXCLUDE_KEYWORDS: tuple[str, ...] = (
    "for-sale", "price-reduced", "prop-for-sale", "engine-for-sale",
    "classified", "classifieds",
    "fs", "wtb", "wts", "sold",
)

GULF_KEYWORDS: tuple[str, ...] = (
    "gulf-of-mexico", "gulf of mexico", "gulf of america",
    "gulf-coast", "gulf coast", "gulf-states", "gulf states", "gulf region",
    "northern-gulf", "eastern-gulf", "western-gulf",
    "florida", "alabama", "mississippi", "louisiana", "texas",
    "panhandle", "big-bend", "tampa-bay", "tampa bay", "charlotte-harbor",
    "apalachicola", "pensacola", "mobile-bay", "mobile bay", "biloxi",
    "destin", "venice-louisiana", "port-fourchon", "galveston",
    "port-aransas", "south-padre", "corpus-christi", "freeport",
    "key-west", "florida-keys", "florida keys", "fort-myers",
    "ten-thousand-islands", "everglades",
    "gulf-council", "gulf council", "gmfmc",
    "red-snapper", "red snapper",
    "gag-grouper", "gag grouper", "red-grouper", "red grouper",
    "black-grouper", "scamp-grouper", "scamp grouper", "yellowedge-grouper",
    "snowy-grouper", "warsaw-grouper", "speckled-hind",
    "vermilion-snapper", "vermilion snapper",
    "gray-triggerfish", "grey-triggerfish", "triggerfish",
    "greater-amberjack", "lesser-amberjack", "amberjack",
    "banded-rudderfish", "almaco-jack",
    "hogfish", "mutton-snapper", "mutton snapper",
    "yellowtail-snapper", "yellowtail snapper",
    "lane-snapper", "queen-snapper", "silk-snapper", "schoolmaster-snapper",
    "blueline-tilefish", "golden-tilefish", "tilefish",
    "king-mackerel", "king mackerel", "kingfish",
    "spanish-mackerel", "spanish mackerel",
    "cobia", "mahi", "dolphinfish",
    "spiny-lobster", "stone-crab", "stone crab",
    "gulf-shrimp", "brown-shrimp", "white-shrimp", "pink-shrimp", "royal-red-shrimp",
    "tarpon", "snook", "redfish", "red-drum", "red drum",
    "speckled-trout", "spotted-seatrout", "spotted seatrout",
    "tripletail", "sheepshead", "pompano",
    "red-tide", "deepwater-horizon", "dead-zone", "hypoxia",
    "loop-current",
)

WEAK_POLICY_KEYWORDS: tuple[str, ...] = (
    "regulation", "regulations",
    "closure", "reopen",
    "noaa", "fwc", "tpwd", "ldwf", "lwf", "wlf", "sefsc", "nmfs",
    "enforcement", "poaching",
)

WEAK_GULF_KEYWORDS: tuple[str, ...] = (
    "florida", "fl", "alabama", "al", "mississippi", "ms",
    "louisiana", "la", "texas", "tx",
)

_POLICY_SET  = frozenset(k.lower() for k in POLICY_KEYWORDS)
_EXCLUDE_SET = frozenset(k.lower() for k in EXCLUDE_KEYWORDS)
_GULF_SET    = frozenset(k.lower() for k in GULF_KEYWORDS)
_WEAK_POLICY = frozenset(k.lower() for k in WEAK_POLICY_KEYWORDS)
_WEAK_GULF   = frozenset(k.lower() for k in WEAK_GULF_KEYWORDS)
_STRONG_POLICY_SET = _POLICY_SET - _WEAK_POLICY
_STRONG_GULF_SET   = _GULF_SET   - _WEAK_GULF
_WORDSPLIT_RE = re.compile(r"[^a-z0-9]+")


def _tokens(s: str) -> list[str]:
    return [t for t in _WORDSPLIT_RE.split(s.lower()) if t]


def _matches_keyword(kw: str, text_lower: str, token_set: set[str]) -> bool:
    kw_toks = _tokens(kw)
    if not kw_toks:
        return False
    if len(kw_toks) == 1:
        return kw_toks[0] in token_set
    phrase = "-".join(kw_toks)
    normalized = "-".join(_tokens(text_lower))
    return phrase in normalized


def policy_hits(text: str) -> list[str]:
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    if any(_matches_keyword(bad, s, toks) for bad in _EXCLUDE_SET):
        return []
    return [kw for kw in _POLICY_SET if _matches_keyword(kw, s, toks)]


def gulf_hits(text: str) -> list[str]:
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    return [kw for kw in _GULF_SET if _matches_keyword(kw, s, toks)]

## Location Detection

Infers the dominant Gulf state (FL / AL / MS / LA / TX) from the transcript text, falling back to `Gulf Council`.

In [ ]:
_STATE_SIGNALS: dict[str, tuple[str, ...]] = {
    "Florida": (
        "florida", "fl",
        "pensacola", "tampa", "miami", "jacksonville", "tallahassee",
        "destin", "key-west", "key west", "florida-keys", "florida keys",
        "charlotte-harbor", "charlotte harbor", "apalachicola",
        "panama-city", "panama city", "st-petersburg", "st. petersburg",
        "fort-myers", "fort myers", "naples", "sarasota", "clearwater",
        "everglades", "panhandle", "big-bend", "big bend", "tampa-bay",
        "tampa bay", "ten-thousand-islands", "ten thousand islands",
        "fwc", "myfwc",
    ),
    "Alabama": (
        "alabama", "al",
        "mobile", "mobile-bay", "mobile bay",
        "gulf-shores", "gulf shores", "orange-beach", "orange beach",
        "dauphin-island", "dauphin island", "fort-morgan", "fort morgan",
        "outdoor-alabama",
    ),
    "Mississippi": (
        "mississippi", "ms",
        "biloxi", "gulfport", "pascagoula", "ocean-springs", "ocean springs",
        "bay-st-louis", "bay st louis", "long-beach", "long beach",
        "dmrms", "msdmr",
    ),
    "Louisiana": (
        "louisiana", "la",
        "new-orleans", "new orleans", "venice-louisiana", "venice louisiana",
        "port-fourchon", "port fourchon", "grand-isle", "grand isle",
        "lake-charles", "lake charles", "baton-rouge", "baton rouge",
        "houma", "lafayette", "lwf", "ldwf", "wlf",
    ),
    "Texas": (
        "texas", "tx",
        "galveston", "corpus-christi", "corpus christi",
        "port-aransas", "port aransas", "south-padre", "south padre",
        "freeport", "houston", "brownsville", "matagorda",
        "rockport", "tpwd",
    ),
}


def extract_state(text: str) -> str:
    """Pick the dominant Gulf state mentioned in `text`. Returns one of
    {Florida, Alabama, Mississippi, Louisiana, Texas} when a state can be
    confidently identified, else 'Gulf Council'."""
    if not text:
        return "Gulf Council"
    s = text.lower()
    toks = set(_tokens(s))
    counts: dict[str, int] = {}
    first_pos: dict[str, int] = {}
    for state, signals in _STATE_SIGNALS.items():
        c = 0
        fp = -1
        for sig in signals:
            if _matches_keyword(sig, s, toks):
                c += 1
                idx = s.find(sig.replace("-", " "))
                if idx < 0:
                    idx = s.find(sig)
                if idx >= 0 and (fp < 0 or idx < fp):
                    fp = idx
        if c:
            counts[state] = c
            first_pos[state] = fp
    if not counts:
        return "Gulf Council"
    max_c = max(counts.values())
    leaders = [st for st, c in counts.items() if c == max_c]
    if len(leaders) == 1:
        return leaders[0]
    return min(leaders, key=lambda st: first_pos[st])

## Configuration — Channels, Queries & Targets

The YouTube `CHANNELS` to crawl, the `SEARCH_QUERIES`, the download target, worker counts, and the optional cookies file used to avoid YouTube IP blocks.

In [ ]:

CHANNELS: list[dict] = [
    # Management bodies
    {"slug": "gmfmc", "url": "https://www.youtube.com/@GulfCouncil",          "name": "Gulf of Mexico Fishery Management Council"},
    {"slug": "noaa",  "url": "https://www.youtube.com/@NOAAFisheries",        "name": "NOAA Fisheries"},
    {"slug": "safmc", "url": "https://www.youtube.com/@SouthAtlanticCouncil", "name": "South Atlantic Fishery Management Council"},
    {"slug": "gsmfc", "url": "https://www.youtube.com/@gsmfc",                "name": "Gulf States Marine Fisheries Commission"},
    # State agencies
    {"slug": "fwc",   "url": "https://www.youtube.com/@FloridaFishandWildlife","name": "Florida FWC"},
    {"slug": "tpwd",  "url": "https://www.youtube.com/@TexasParksWildlife",   "name": "Texas Parks & Wildlife"},
    {"slug": "lawlf", "url": "https://www.youtube.com/@LouisianaWildlifeFisheries","name": "Louisiana Wildlife & Fisheries"},
    {"slug": "msdmr", "url": "https://www.youtube.com/@msdmr",                "name": "Mississippi DMR"},
    {"slug": "oal",   "url": "https://www.youtube.com/@OutdoorAlabamaOfficial","name": "Outdoor Alabama"},
    # Advocacy / NGO
    {"slug": "oc",    "url": "https://www.youtube.com/@OceanConservancy",     "name": "Ocean Conservancy"},
    {"slug": "edf",   "url": "https://www.youtube.com/@EnvDefenseFund",       "name": "Environmental Defense Fund"},
    {"slug": "pew",   "url": "https://www.youtube.com/@PewCharitableTrusts",  "name": "Pew Charitable Trusts"},
    {"slug": "cca",   "url": "https://www.youtube.com/@CCAFlorida",           "name": "CCA Florida"},
    {"slug": "ccala", "url": "https://www.youtube.com/@CCALouisiana",         "name": "CCA Louisiana"},
    {"slug": "ccatx", "url": "https://www.youtube.com/@CCATexas",             "name": "CCA Texas"},
    {"slug": "hgulf", "url": "https://www.youtube.com/@HealthyGulf",          "name": "Healthy Gulf"},
    {"slug": "btt",   "url": "https://www.youtube.com/@BonefishTarponTrust",  "name": "Bonefish & Tarpon Trust"},
    {"slug": "sra",   "url": "https://www.youtube.com/@GulfReefFishShareholdersAlliance",
                                                                              "name": "Gulf Reef Fish Shareholders Alliance"},
    {"slug": "wild",  "url": "https://www.youtube.com/@WildOceans",           "name": "Wild Oceans"},
    {"slug": "trcp",  "url": "https://www.youtube.com/channel/UCxvwuxKu9s68TXLpK3ZUe8g",
                                                                              "name": "Theodore Roosevelt Conservation Partnership"},
    {"slug": "asa",   "url": "https://www.youtube.com/channel/UC4QbseKNF1PD9KqhH8PJJtw",
                                                                              "name": "American Sportfishing Association"},
    {"slug": "aswg",  "url": "https://www.youtube.com/channel/UCGMeeAhU1o_ASSxQ4y37H7Q",
                                                                              "name": "American Saltwater Guides Association"},
    # Sea Grant
    {"slug": "flsg",  "url": "https://www.youtube.com/@FloridaSeaGrant",      "name": "Florida Sea Grant"},
    {"slug": "txsg",  "url": "https://www.youtube.com/@TexasSeaGrant",        "name": "Texas Sea Grant"},
    {"slug": "lasg",  "url": "https://www.youtube.com/@LouisianaSeaGrantCollegeProgram",
                                                                              "name": "Louisiana Sea Grant"},
    {"slug": "masgc", "url": "https://www.youtube.com/@masgc",                "name": "Mississippi-Alabama Sea Grant Consortium"},
    # Science / research institutions — Gulf stock science, red tide, habitat
    {"slug": "mote",  "url": "https://www.youtube.com/@MoteMarineLab",        "name": "Mote Marine Laboratory"},
    {"slug": "disl",  "url": "https://www.youtube.com/channel/UCphmeb5XTCoBhzk2b937_Rw",
                                                                              "name": "Dauphin Island Sea Lab"},
    {"slug": "flmus", "url": "https://www.youtube.com/@floridamuseum",        "name": "Florida Museum"},
    {"slug": "afs",   "url": "https://www.youtube.com/channel/UCzvdtkj-NqaPruJYJNzExBA",
                                                                              "name": "American Fisheries Society"},
    {"slug": "ccan",  "url": "https://www.youtube.com/channel/UCcqlfw5YwsSeil0iXyK3iDw",
                                                                              "name": "Coastal Conservation Association"},
    {"slug": "ufifas","url": "https://www.youtube.com/@UFIFAS",               "name": "UF/IFAS"},
    # Trade press / industry
    {"slug": "nf",    "url": "https://www.youtube.com/@nationalfisherman",    "name": "National Fisherman"},
    {"slug": "salt",  "url": "https://www.youtube.com/@SaltStrong",           "name": "Salt Strong"},
    {"slug": "sfmag", "url": "https://www.youtube.com/@SportFishingMag",      "name": "Sport Fishing Magazine"},
    {"slug": "infish","url": "https://www.youtube.com/channel/UCXLcsCBsiery-u-rZus2hXQ",
                                                                              "name": "In-Fisherman"},
    {"slug": "wptv",  "url": "https://www.youtube.com/@WaypointTV",           "name": "Waypoint TV"},
    # ── Angler / charter / regional media (broad — strict filter does the
    #    relevance work; these surface one-off Gulf regulation episodes) ──
    {"slug": "blacktiph", "url": "https://www.youtube.com/@BlacktipH",        "name": "BlacktipH"},
    {"slug": "hubbards",  "url": "https://www.youtube.com/@HubbardsMarina",   "name": "Hubbard's Marina"},
    {"slug": "addictive", "url": "https://www.youtube.com/@AddictiveFishing", "name": "Addictive Fishing"},
    {"slug": "fsftv",  "url": "https://www.youtube.com/channel/UCpnyYJ_9YYzYdeZNBNqzXZA",
                                                                              "name": "Florida Sport Fishing TV"},
    {"slug": "tsfmag", "url": "https://www.youtube.com/@TexasSaltwaterFishingMagazine",
                                                                              "name": "Texas Saltwater Fishing Magazine"},
    {"slug": "lasptv", "url": "https://www.youtube.com/channel/UCw6TwJr8rHTm8ddMnS9OTLw",
                                                                              "name": "Louisiana Sportsman"},
    {"slug": "marshman","url": "https://www.youtube.com/@MarshManMasson",     "name": "Marsh Man Masson"},


    # fisheries, charter regulations) — high transcript yield per video.
    {"slug": "pod-noaa", "url": "https://www.youtube.com/channel/UCryOAXMZlwj8coCVIcPJ5EA",
                                                            "name": "NOAA Fisheries Podcasts"},
    {"slug": "pod-hri",  "url": "https://www.youtube.com/channel/UCJE1jV3oKjvcbAIEDWKpjrA",
                                                            "name": "Harte Research Institute — Gulf Stream Podcast"},
    {"slug": "pod-mfcn", "url": "https://www.youtube.com/channel/UCCQ_7IUIiJmOkJ0k2DHmeHw",
                                                            "name": "Marine Fish Conservation Network"},
    {"slug": "pod-nru",  "url": "https://www.youtube.com/channel/UCZ-uehW5nyQ8uZMDSHG_low",
                                                            "name": "Natural Resources University — Fish University"},
    {"slug": "pod-ccw",  "url": "https://www.youtube.com/channel/UCl3KJIWsGhHI4zwvKHAouaw",
                                                            "name": "Captains For Clean Water"},
    {"slug": "pod-dfg",  "url": "https://www.youtube.com/channel/UC0g9Yty_DUQsVRsjquRgqog",
                                                            "name": "Dispatches from the Gulf"},
    {"slug": "pod-mso",  "url": "https://www.youtube.com/channel/UCJmy0a8O_evm1iFS-h4mAiw",
                                                            "name": "Mississippi Outdoors"},
    {"slug": "pod-gcfr", "url": "https://www.youtube.com/channel/UC6bhYtZhMwHX4YfTIYwWqIg",
                                                            "name": "Gulf Coast Fishing Report"},
    {"slug": "pod-se",   "url": "https://www.youtube.com/channel/UC5easOoo4XbG5yd8eZ_gXIw",
                                                            "name": "Saltwater Experience — Tom Rowland"},
    {"slug": "pod-tsa",  "url": "https://www.youtube.com/channel/UCokqddEmgyrHRku_O9MfBPg",
                                                            "name": "The Sustainable Angler"},
    {"slug": "pod-mh",   "url": "https://www.youtube.com/channel/UC7jzsD4lKviI8HYB3c5lAjA",
                                                            "name": "Mill House Podcast"},
    {"slug": "pod-sb",   "url": "https://www.youtube.com/channel/UCD5h3oNDDLS8vImlyZ02ahg",
                                                            "name": "SeaBros Fishing Podcast"},

    # ── Additional podcast channels (discovered via Gulf-policy search) ─
    {"slug": "pod-alp",  "url": "https://www.youtube.com/channel/UCJYYt1yIkFQ052t6m9mT0FA",
                                                            "name": "Anchor Lines Podcast"},
    {"slug": "pod-ofp",  "url": "https://www.youtube.com/channel/UCrnu6q3XhU9VK-MCURo68xg",
                                                            "name": "Our Florida Podcast"},
    {"slug": "pod-nmw",  "url": "https://www.youtube.com/channel/UCroWPW0Wg6zZPyt42N89P7g",
                                                            "name": "NOAA Making Waves Podcast"},
    {"slug": "pod-flsp", "url": "https://www.youtube.com/channel/UCYBCc4dEcBpkMAYBOYJURtQ",
                                                            "name": "Florida Sportsman"},
    {"slug": "pod-rm",   "url": "https://www.youtube.com/channel/UCKXKgV9Vvd4_2VjdwstRrHg",
                                                            "name": "Capt. Rick Murphy / Florida Insider Fishing Report"},
]


SEARCH_QUERIES: list[str] = [
    "gulf of mexico fishery management council",
    "gulf council red snapper",
    "magnuson stevens act gulf",
    "gulf reef fish amendment",
    "gulf red snapper season",
    "gulf council public meeting",
    "gulf bycatch reduction",
    "gulf for-hire charter",
    "gulf grouper closure",
    "gulf cobia amendment",
    "individual fishing quota gulf",
    "gulf shrimp fishery management",
    "gulf stone crab regulations",
    "gulf council testimony",
    "gulf descending device barotrauma",
    # ── PODCAST-targeted queries (catch independent podcast episodes) ───
    "gulf fishing podcast red snapper",
    "gulf council podcast",
    "florida fishing podcast regulations",
    "louisiana charter captain podcast",
    "texas saltwater podcast snapper",
    "fisheries podcast gulf of mexico",
    "salt strong podcast snapper",
    "captain podcast gulf grouper",
    "gulf reef fish podcast",
    "noaa fisheries podcast gulf",
    "saltwater podcast red snapper",
    "fishing podcast bag limit closure gulf",
]

# How many videos to enumerate per channel and per query.
MAX_VIDEOS_PER_CHANNEL = 80
MAX_VIDEOS_PER_QUERY   = 60

# Default total target (passing-filter videos saved across the whole run).
TARGET_VIDEOS = 6000

TRANSCRIPT_WORKERS = 8

COOKIES_FILE = Path.cwd() / "cookies.txt"

## Paths & Metadata Schema

Output locations under the shared `Data_Repository` and the seven-field metadata schema written for every saved video.

In [ ]:
ROOT         = Path.cwd().resolve()
DATA         = ROOT / "Data_Repository"
RAW_TXT_DIR  = DATA / "scraped_text" / "youtube_podcast"
METADATA_DIR = DATA / "metadata" / "youtube_podcast"
RECORDS_DIR  = DATA / "records"
STATE_FILE   = DATA / "youtube_state.json"

METADATA_FIELDS = (
    "Document_ID", "Source_Type", "Source_Name", "URL", "Title",
    "Published_Date", "Location",
)

## Dependency Check

Verifies the optional heavy dependencies (yt-dlp, youtube-transcript-api) are importable before a run starts.

In [ ]:
try:
    import yt_dlp
    _YTDLP = True
except ImportError:
    _YTDLP = False

try:
    from youtube_transcript_api import YouTubeTranscriptApi
    from youtube_transcript_api._errors import (
        TranscriptsDisabled, NoTranscriptFound, VideoUnavailable,
    )
    _YTAPI = True
except ImportError:
    _YTAPI = False


def _require_deps():
    missing = []
    if not _YTDLP: missing.append("yt-dlp")
    if not _YTAPI: missing.append("youtube-transcript-api")
    if missing:
        sys.exit(f"Missing deps: {missing}. Run: pip install -U {' '.join(missing)}")

## Video Enumeration

Lists recent videos from each channel's uploads and from keyword searches, using yt-dlp's fast flat extractor.

In [ ]:
_YDL_OPTS_FLAT = {
    "quiet": True,
    "no_warnings": True,
    "extract_flat": "in_playlist",   # don't fetch each video — just list them
    "skip_download": True,
    "ignoreerrors": True,
    "playlistend": 200,
}


def enumerate_channel(url: str, limit: int) -> list[dict]:
    """Return [{id, title, upload_date, channel}, ...] for the channel's
    uploads. Uses yt-dlp's flat extraction so we don't fetch every video."""
    opts = dict(_YDL_OPTS_FLAT, playlistend=limit)
    # /videos suffix forces yt-dlp to the uploads playlist for handle URLs.
    feed_url = url.rstrip("/") + "/videos"
    out: list[dict] = []
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(feed_url, download=False)
    except Exception as e:
        print(f"  [enum] {url}: {type(e).__name__}: {e}")
        return out
    if not info:
        return out
    entries = info.get("entries") or []
    channel_name = info.get("channel") or info.get("uploader") or ""
    for e in entries[:limit]:
        if not e or not e.get("id"):
            continue
        out.append({
            "id":            e["id"],
            "title":         (e.get("title") or "").strip(),
            "upload_date":   e.get("upload_date") or "",
            "channel":       e.get("channel") or channel_name,
            "url":           f"https://www.youtube.com/watch?v={e['id']}",
        })
    return out


def search_query(query: str, limit: int) -> list[dict]:
    """yt-dlp's ytsearchN: prefix performs a YouTube search and returns
    flat metadata for the top N results."""
    opts = dict(_YDL_OPTS_FLAT)
    out: list[dict] = []
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(f"ytsearch{limit}:{query}", download=False)
    except Exception as e:
        print(f"  [search] {query!r}: {type(e).__name__}: {e}")
        return out
    if not info:
        return out
    for e in (info.get("entries") or [])[:limit]:
        if not e or not e.get("id"):
            continue
        out.append({
            "id":            e["id"],
            "title":         (e.get("title") or "").strip(),
            "upload_date":   e.get("upload_date") or "",
            "channel":       e.get("channel") or e.get("uploader") or "",
            "url":           f"https://www.youtube.com/watch?v={e['id']}",
        })
    return out

## Transcript Fetching

Pulls each video's captions, parses VTT into plain text, and returns a clear outcome code when captions are missing, disabled or IP-blocked.

In [ ]:
# Outcome tags for fetch_transcript — used by the runner to report exactly
# WHY a video produces no doc (transcript missing vs filter rejection).
TRANSCRIPT_OK            = "ok"
TRANSCRIPT_NONE_DISABLED = "disabled"     # captions turned off by uploader
TRANSCRIPT_NONE_NOT_FOUND= "not_found"    # no caption track at all
TRANSCRIPT_NONE_VIDEO_NA = "video_na"     # video private / unavailable
TRANSCRIPT_NONE_ERR      = "error"        # any other error
TRANSCRIPT_IPBLOCK       = "ip_blocked"   # YouTube blocking our IP
TRANSCRIPT_TOO_SHORT     = "too_short"    # captions exist but body < 400 ch


def _parse_vtt(content: str) -> str:
    """Strip VTT/SRT cue markers and timestamps, return concatenated text."""
    lines: list[str] = []
    seen: set[str] = set()
    for ln in content.split("\n"):
        ln = ln.strip()
        if not ln:
            continue
        if ln.startswith("WEBVTT") or ln.startswith("NOTE"):
            continue
        if "-->" in ln:
            continue
        if ln.isdigit():
            continue
        if ln.startswith("Kind:") or ln.startswith("Language:"):
            continue
        # Deduplicate — VTT auto-captions repeat overlapping cues heavily.
        if ln in seen:
            continue
        seen.add(ln)
        lines.append(ln)
    return " ".join(lines).strip()


def _fetch_transcript_ytdlp(video_id: str) -> tuple[str, str]:
    """Use yt-dlp to download subtitles. Optionally uses COOKIES_FILE for
    authentication when YouTube IP-blocks unauthenticated requests."""
    import tempfile
    url = f"https://www.youtube.com/watch?v={video_id}"
    with tempfile.TemporaryDirectory() as td:
        opts = {
            "quiet": True,
            "no_warnings": True,
            "skip_download": True,
            "writesubtitles": True,
            "writeautomaticsub": True,
            "subtitleslangs": ["en", "en-US", "en-GB", "en.*"],
            "subtitlesformat": "vtt/best",
            "outtmpl": str(Path(td) / "%(id)s.%(ext)s"),
        }
        if COOKIES_FILE.exists():
            opts["cookiefile"] = str(COOKIES_FILE)
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:  # type: ignore[arg-type]
                ydl.download([url])
        except Exception as e:
            # yt-dlp raises yt_dlp.utils.DownloadError but the public stub
            # doesn't expose it; class-name check avoids the linter
            # complaint while still distinguishing the failure modes.
            msg = str(e).lower()
            if "sign in" in msg or "not a bot" in msg:
                return "", TRANSCRIPT_IPBLOCK
            if "unavailable" in msg or "private" in msg:
                return "", TRANSCRIPT_NONE_VIDEO_NA
            return "", TRANSCRIPT_NONE_ERR
        # Find the best subtitle file. Prefer manual EN tracks over auto.
        candidates = sorted(p for p in Path(td).iterdir()
                            if p.suffix in (".vtt", ".srt"))
        manual = [p for p in candidates if "auto" not in p.name.lower()]
        ordered = (manual or candidates)
        for p in ordered:
            try:
                content = p.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue
            text = _parse_vtt(content)
            if text:
                return text, TRANSCRIPT_OK
        return "", TRANSCRIPT_NONE_NOT_FOUND


def fetch_transcript(video_id: str) -> tuple[str, str]:
    """Return (text, outcome). Tries youtube-transcript-api first; if that
    returns an IP-block or generic error, falls back to yt-dlp subtitle
    download (which uses different YouTube endpoints and can use cookies
    via COOKIES_FILE)."""
    try:
        from youtube_transcript_api._errors import IpBlocked  # type: ignore
    except ImportError:
        IpBlocked = None  # type: ignore
    # 1) Try youtube-transcript-api — fast path when not blocked.
    try:
        ytt = YouTubeTranscriptApi()
        try:
            transcript = ytt.fetch(video_id, languages=["en", "en-US", "en-GB"])
            text = " ".join(seg.text for seg in transcript if seg.text).strip()
            if text:
                return text, TRANSCRIPT_OK
        except NoTranscriptFound:
            pass
        except TranscriptsDisabled:
            # Captions disabled — yt-dlp can't help either.
            return "", TRANSCRIPT_NONE_DISABLED
        except VideoUnavailable:
            return "", TRANSCRIPT_NONE_VIDEO_NA
        except Exception as e:
            # IP block or generic error → try yt-dlp fallback.
            if IpBlocked is not None and isinstance(e, IpBlocked):
                pass  # fall through to yt-dlp
            elif "ipblocked" in type(e).__name__.lower():
                pass
            # else fall through anyway — yt-dlp often succeeds when API fails.
    except Exception:
        pass
    # 2) yt-dlp fallback.
    return _fetch_transcript_ytdlp(video_id)

## Output & CSV

Writes each saved video's `.txt` and `.json`, and rebuilds the consolidated `youtube.csv` index from all metadata files.

In [ ]:
def render_doc_txt(doc: dict) -> str:
    lines = [
        f"Document_ID  : {doc['Document_ID']}",
        f"Source_Type  : {doc['Source_Type']}",
        f"Source_Name  : {doc['Source_Name']}",
        f"URL          : {doc['URL']}",
        f"Title        : {doc['Title']}",
        f"Published    : {doc.get('Published_Date','')}",
        f"Location     : {doc['Location']}",
        "=" * 80,
        "",
        doc.get("_text", ""),
    ]
    return "\n".join(lines)


def write_doc(doc: dict):
    tid = doc["Document_ID"]
    (RAW_TXT_DIR / f"{tid}.txt").write_text(render_doc_txt(doc), encoding="utf-8")
    meta = {k: doc.get(k, "") for k in METADATA_FIELDS}
    (METADATA_DIR / f"{tid}.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
    )


def rebuild_csv() -> int:
    """Read every metadata JSON and rebuild youtube.csv."""
    csv_path = RECORDS_DIR / "youtube_podcast.csv"
    rows: list[dict] = []
    for jf in sorted(METADATA_DIR.glob("*.json")):
        try:
            d = json.loads(jf.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(d, dict) or "Document_ID" not in d:
            continue
        rows.append({k: d.get(k, "") for k in METADATA_FIELDS})
    if rows:
        with csv_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(METADATA_FIELDS),
                               quoting=csv.QUOTE_ALL)
            w.writeheader()
            w.writerows(rows)
    return len(rows)

## Run State

Loads and saves resumable progress (seen video IDs, per-channel stats) in `youtube_state.json`.

In [ ]:
def load_state() -> dict:
    if STATE_FILE.exists():
        try:
            return json.loads(STATE_FILE.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {"seen_video_ids": [], "channel_stats": {}, "query_stats": {}}


def save_state(s: dict):
    STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    STATE_FILE.write_text(json.dumps(s, indent=1, ensure_ascii=False),
                          encoding="utf-8")

## Filter, 10-Year Window & Save

Normalises the upload date, applies the **last-10-years** window (`within_data_window`), runs the strict policy + Gulf gate, and builds the metadata record for videos that pass.

In [ ]:
def _normalize_upload_date(d: str) -> str:
    """yt-dlp gives YYYYMMDD. Return ISO YYYY-MM-DD or '' on bad input."""
    if not d or len(d) != 8 or not d.isdigit():
        return ""
    return f"{d[:4]}-{d[4:6]}-{d[6:8]}"


DATA_WINDOW_YEARS = 10

def within_data_window(date_str: str, years: int = DATA_WINDOW_YEARS) -> bool:
    """True if `date_str` (ISO YYYY-MM-DD or similar) falls within the last
    `years` years. Reads the first 4-digit year; empty / unparseable dates
    return True so we never drop media we can't date."""
    if not date_str:
        return True
    m = re.search(r"(?:19|20)\d{2}", date_str)
    if not m:
        return True
    return int(m.group(0)) >= datetime.now().year - years



def process_video(v: dict, src_slug: str, src_name: str
                  ) -> tuple[Optional[dict], str]:
    """Fetch transcript, apply the strict Gulf+policy gate.
    Returns (doc, outcome). outcome is one of:
        'saved'      — passed all filters and is being saved
        'disabled'   — captions disabled by uploader
        'not_found'  — no caption track available
        'video_na'   — video private / unavailable
        'error'      — transcript fetch raised
        'too_short'  — transcript exists but < 400 chars
        'no_policy'  — failed policy keyword gate
        'no_gulf'    — failed Gulf keyword gate
        'no_strong'  — only weak keywords matched
    """
    # Last-N-years window: skip stale uploads before fetching transcripts.
    pub = _normalize_upload_date(v.get("upload_date", ""))
    if not within_data_window(pub):
        return None, "too_old"
    text, t_outcome = fetch_transcript(v["id"])
    if t_outcome != TRANSCRIPT_OK:
        return None, t_outcome
    if len(text) < 400:
        return None, TRANSCRIPT_TOO_SHORT
    blob = f"{v['title']}\n{text}"
    phits = policy_hits(blob)
    if not phits:
        return None, "no_policy"
    ghits = gulf_hits(blob)
    if not ghits:
        return None, "no_gulf"
    strong_p = [h for h in phits if h in _STRONG_POLICY_SET]
    strong_g = [h for h in ghits if h in _STRONG_GULF_SET]
    if not (strong_p and strong_g):
        return None, "no_strong"
    doc_id = f"yt_{src_slug}_{v['id']}"
    doc = {
        "Document_ID":    doc_id,
        "Source_Type":    "youtube",
        "Source_Name":    src_name,
        "URL":            v["url"],
        "Title":          v["title"][:500],
        "Published_Date": pub,
        "Location":       extract_state(blob),
        "_text":          text,
    }
    return doc, "saved"

## Run — YouTube Crawl

Orchestrates Part 1: enumerate candidates → fetch transcripts in parallel → filter (policy + Gulf + 10-year) → save → rebuild the CSV, with a per-outcome breakdown.

In [ ]:
def run(target_total: int = TARGET_VIDEOS,
        only_channels: str = "",
        skip_search: bool = False,
        rebuild_csv_only: bool = False) -> int:
    _require_deps()

    for p in (DATA, RAW_TXT_DIR, METADATA_DIR, RECORDS_DIR):
        p.mkdir(parents=True, exist_ok=True)

    if rebuild_csv_only:
        n = rebuild_csv()
        print(f"Rebuilt youtube_podcast.csv with {n} rows.")
        return n

    state = load_state()
    seen: set[str] = set(state.get("seen_video_ids", []))
    only_set = {s.strip() for s in only_channels.split(",") if s.strip()}

    print(f"[cfg] target_total     = {target_total}")
    print(f"[cfg] channels         = {len(CHANNELS)}")
    print(f"[cfg] search queries   = {0 if skip_search else len(SEARCH_QUERIES)}")
    print(f"[cfg] transcript workers = {TRANSCRIPT_WORKERS}")
    print(f"[cfg] already-seen IDs in state = {len(seen)}")
    print()

    # ── Phase 1: enumerate candidate videos across channels + searches.
    candidates: list[tuple[dict, str, str]] = []  # (video, slug, src_name)

    for ch in CHANNELS:
        if only_set and ch["slug"] not in only_set:
            continue
        print(f"[enum] channel {ch['name']:<45}  {ch['url']}")
        videos = enumerate_channel(ch["url"], MAX_VIDEOS_PER_CHANNEL)
        new = [v for v in videos if v["id"] not in seen]
        print(f"       found {len(videos)} videos, {len(new)} new")
        for v in new:
            candidates.append((v, ch["slug"], ch["name"]))
        state["channel_stats"][ch["slug"]] = {
            "url": ch["url"], "total_listed": len(videos), "new": len(new),
            "at": datetime.now(timezone.utc).isoformat(),
        }

    if not skip_search:
        for q in SEARCH_QUERIES:
            print(f"[search] {q!r}")
            videos = search_query(q, MAX_VIDEOS_PER_QUERY)
            new = [v for v in videos if v["id"] not in seen]
            print(f"         found {len(videos)} hits, {len(new)} new")
            for v in new:
                # Search results: slug = "search", name = channel from result
                candidates.append((v, "search", v["channel"] or "YouTube"))
            state["query_stats"][q] = {
                "total_listed": len(videos), "new": len(new),
                "at": datetime.now(timezone.utc).isoformat(),
            }

    # Dedupe candidates by id (search + channel might overlap).
    dedup: dict[str, tuple[dict, str, str]] = {}
    for v, slug, name in candidates:
        if v["id"] not in dedup:
            dedup[v["id"]] = (v, slug, name)
    queue = list(dedup.values())
    print()
    print(f"[queue] {len(queue)} unique candidate videos to transcript-test")

    # ── Phase 2: fetch transcripts in parallel, filter, save passing docs.
    saved = 0
    examined = 0
    outcomes: dict[str, int] = {}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=TRANSCRIPT_WORKERS) as pool:
        futures = {
            pool.submit(process_video, v, slug, name): v["id"]
            for v, slug, name in queue
        }
        for fut in as_completed(futures):
            vid = futures[fut]
            seen.add(vid)
            examined += 1
            try:
                doc, outcome = fut.result()
            except Exception:
                outcomes["worker_error"] = outcomes.get("worker_error", 0) + 1
                continue
            outcomes[outcome] = outcomes.get(outcome, 0) + 1
            if doc is not None:
                write_doc(doc)
                saved += 1
                safe_title = doc["Title"][:70].encode("ascii", "replace").decode("ascii")
                print(f"  [{saved:>4}/{target_total}] {doc['Document_ID']:<32} "
                      f"loc={doc['Location']:<13}  {safe_title}")
            if saved >= target_total:
                break
            if examined % 50 == 0:
                rate = examined / max(time.time() - t0, 0.1)
                print(f"  [progress] examined={examined} saved={saved}  ({rate:.1f}/s)")

    state["seen_video_ids"] = sorted(seen)
    save_state(state)
    total_rows = rebuild_csv()
    elapsed = time.time() - t0
    print()
    print(f"=== youtube_podcast.csv rebuilt: {total_rows} total rows ===")
    print(f"=== this run: examined {examined} videos, saved {saved} in {elapsed:.1f}s ===")
    print("=== outcome breakdown (why each video did/didn't save) ===")
    # Order outcomes from most → least useful for readability.
    order = ["saved", "too_old", "no_strong", "no_gulf", "no_policy", "too_short",
             "not_found", "disabled", "video_na", "ip_blocked",
             "error", "worker_error"]
    for k in order:
        if k in outcomes:
            print(f"     {outcomes[k]:>5}  {k}")
    for k, v in outcomes.items():
        if k not in order:
            print(f"     {v:>5}  {k}")
    if outcomes.get("ip_blocked", 0) >= max(5, examined // 10):
        print()
        print("WARNING: YouTube is IP-blocking transcript requests. To fix:")
        print("  1) Install a browser extension like 'Get cookies.txt LOCALLY'.")
        print("  2) Log into youtube.com in your browser.")
        print("  3) Export cookies as 'cookies.txt' in this project root.")
        print(f"  4) Confirmed location: {COOKIES_FILE}")
        print("  5) Re-run the pipeline.")
    return total_rows


def main():
    ap = argparse.ArgumentParser(description="YouTube Gulf-Council / FEI scraper")
    ap.add_argument("-n", "--target-videos", type=int, default=TARGET_VIDEOS,
                    help=f"target number of saved videos (default {TARGET_VIDEOS})")
    ap.add_argument("--only", type=str, default="",
                    help="comma-separated channel slugs to enumerate")
    ap.add_argument("--skip-search", action="store_true",
                    help="skip the keyword-search phase; only use channels")
    ap.add_argument("--rebuild-csv-only", action="store_true",
                    help="skip fetch, rebuild youtube.csv from existing JSONs")
    args, _unknown = ap.parse_known_args()
    run(target_total=args.target_videos,
        only_channels=args.only,
        skip_search=args.skip_search,
        rebuild_csv_only=args.rebuild_csv_only)


if __name__ == "__main__":
    main()

# Part 2 — Podcast Transcript Scraper

Reads a curated list of fisheries podcast RSS feeds, downloads and transcribes
each episode's audio (HuggingFace API or local Whisper), and saves only
transcripts about Gulf policy published in the **last 10 years** — into the same
`Data_Repository` as Part 1.

## Setup — Install & Imports

Installs the podcast tooling (feedparser, requests, faster-whisper) and imports the standard-library modules used throughout Part 2.

In [ ]:
!pip install -U feedparser requests faster-whisper
from __future__ import annotations

import os
import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*HF_TOKEN.*Colab.*",
)
warnings.filterwarnings(
    "ignore",
    message=r".*unauthenticated requests to the HF Hub.*",
)
# huggingface_hub respects this to skip the Colab-secret probe entirely.
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")

import argparse
import csv
import json
import re
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

## Policy & Gulf Relevance Filter

The same strong/weak policy and Gulf keyword gates as Part 1, applied to each episode's title, description and transcript.

In [ ]:
# Identical to the gate in sources_pipeline.py and youtube_pipeline.py.
# Inlined so this script is self-contained.
POLICY_KEYWORDS: tuple[str, ...] = (
    "gulf-council", "gulf council", "gmfmc", "nmfs", "noaa", "fwc",
    "tpwd", "ldwf", "lwf", "wlf", "sefsc", "magnuson", "msa",
    "amendment", "regulation", "regulations", "rulemaking",
    "ifq", "quota", "allocation", "fmp", "mrip", "mrfss",
    "closure", "reopen", "open-season", "closed-season",
    "season-closure", "season-opening",
    "bag-limit", "size-limit", "slot-limit", "trip-limit", "vessel-limit",
    "stock-assessment", "overfishing", "overfished",
    "for-hire",
    "by-catch", "bycatch", "release-mortality",
    "descending-device", "descender", "venting-tool", "barotrauma",
    "enforcement", "poaching",
    "public-comment", "comment-period", "public-hearing", "public-meeting",
    "testimony",
    "ecosystem-based-management", "ebm", "ecosystem-assessment",
    "fishery-management", "fisheries-management",
    "sustainable-fishery", "sustainable-fisheries",
    "red-tide",
)
EXCLUDE_KEYWORDS: tuple[str, ...] = (
    "for-sale", "price-reduced", "prop-for-sale", "engine-for-sale",
    "classified", "classifieds",
    "fs", "wtb", "wts", "sold",
)
GULF_KEYWORDS: tuple[str, ...] = (
    "gulf-of-mexico", "gulf of mexico", "gulf of america",
    "gulf-coast", "gulf coast", "gulf-states", "gulf states", "gulf region",
    "northern-gulf", "eastern-gulf", "western-gulf",
    "florida", "alabama", "mississippi", "louisiana", "texas",
    "panhandle", "big-bend", "tampa-bay", "tampa bay", "charlotte-harbor",
    "apalachicola", "pensacola", "mobile-bay", "mobile bay", "biloxi",
    "destin", "venice-louisiana", "port-fourchon", "galveston",
    "port-aransas", "south-padre", "corpus-christi", "freeport",
    "key-west", "florida-keys", "florida keys", "fort-myers",
    "ten-thousand-islands", "everglades",
    "gulf-council", "gulf council", "gmfmc",
    "red-snapper", "red snapper",
    "gag-grouper", "gag grouper", "red-grouper", "red grouper",
    "black-grouper", "scamp-grouper", "scamp grouper", "yellowedge-grouper",
    "snowy-grouper", "warsaw-grouper", "speckled-hind",
    "vermilion-snapper", "vermilion snapper",
    "gray-triggerfish", "grey-triggerfish", "triggerfish",
    "greater-amberjack", "lesser-amberjack", "amberjack",
    "banded-rudderfish", "almaco-jack",
    "hogfish", "mutton-snapper", "mutton snapper",
    "yellowtail-snapper", "yellowtail snapper",
    "lane-snapper", "queen-snapper", "silk-snapper", "schoolmaster-snapper",
    "blueline-tilefish", "golden-tilefish", "tilefish",
    "king-mackerel", "king mackerel", "kingfish",
    "spanish-mackerel", "spanish mackerel",
    "cobia", "mahi", "dolphinfish",
    "spiny-lobster", "stone-crab", "stone crab",
    "gulf-shrimp", "brown-shrimp", "white-shrimp", "pink-shrimp", "royal-red-shrimp",
    "tarpon", "snook", "redfish", "red-drum", "red drum",
    "speckled-trout", "spotted-seatrout", "spotted seatrout",
    "tripletail", "sheepshead", "pompano",
    "red-tide", "deepwater-horizon", "dead-zone", "hypoxia",
    "loop-current",
)
WEAK_POLICY_KEYWORDS: tuple[str, ...] = (
    "regulation", "regulations",
    "closure", "reopen",
    "noaa", "fwc", "tpwd", "ldwf", "lwf", "wlf", "sefsc", "nmfs",
    "enforcement", "poaching",
)
WEAK_GULF_KEYWORDS: tuple[str, ...] = (
    "florida", "fl", "alabama", "al", "mississippi", "ms",
    "louisiana", "la", "texas", "tx",
)

_POLICY_SET  = frozenset(k.lower() for k in POLICY_KEYWORDS)
_EXCLUDE_SET = frozenset(k.lower() for k in EXCLUDE_KEYWORDS)
_GULF_SET    = frozenset(k.lower() for k in GULF_KEYWORDS)
_WEAK_POLICY = frozenset(k.lower() for k in WEAK_POLICY_KEYWORDS)
_WEAK_GULF   = frozenset(k.lower() for k in WEAK_GULF_KEYWORDS)
_STRONG_POLICY_SET = _POLICY_SET - _WEAK_POLICY
_STRONG_GULF_SET   = _GULF_SET   - _WEAK_GULF
_WORDSPLIT_RE = re.compile(r"[^a-z0-9]+")


def _tokens(s: str) -> list[str]:
    return [t for t in _WORDSPLIT_RE.split(s.lower()) if t]


def _matches_keyword(kw: str, text_lower: str, token_set: set[str]) -> bool:
    kw_toks = _tokens(kw)
    if not kw_toks:
        return False
    if len(kw_toks) == 1:
        return kw_toks[0] in token_set
    phrase = "-".join(kw_toks)
    normalized = "-".join(_tokens(text_lower))
    return phrase in normalized


def policy_hits(text: str) -> list[str]:
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    if any(_matches_keyword(bad, s, toks) for bad in _EXCLUDE_SET):
        return []
    return [kw for kw in _POLICY_SET if _matches_keyword(kw, s, toks)]


def gulf_hits(text: str) -> list[str]:
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    return [kw for kw in _GULF_SET if _matches_keyword(kw, s, toks)]

## Location Detection

Infers the dominant Gulf state from the episode text, falling back to `Gulf Council`.

In [ ]:
_STATE_SIGNALS: dict[str, tuple[str, ...]] = {
    "Florida": (
        "florida", "fl", "pensacola", "tampa", "miami", "jacksonville", "tallahassee",
        "destin", "key-west", "key west", "florida-keys", "florida keys",
        "charlotte-harbor", "charlotte harbor", "apalachicola",
        "panama-city", "panama city", "st-petersburg", "st. petersburg",
        "fort-myers", "fort myers", "naples", "sarasota", "clearwater",
        "everglades", "panhandle", "big-bend", "big bend", "tampa-bay",
        "tampa bay", "ten-thousand-islands", "ten thousand islands",
        "fwc", "myfwc",
    ),
    "Alabama": (
        "alabama", "al", "mobile", "mobile-bay", "mobile bay",
        "gulf-shores", "gulf shores", "orange-beach", "orange beach",
        "dauphin-island", "dauphin island", "fort-morgan", "fort morgan",
        "outdoor-alabama",
    ),
    "Mississippi": (
        "mississippi", "ms", "biloxi", "gulfport", "pascagoula",
        "ocean-springs", "ocean springs", "bay-st-louis", "bay st louis",
        "long-beach", "long beach", "dmrms", "msdmr",
    ),
    "Louisiana": (
        "louisiana", "la", "new-orleans", "new orleans", "venice-louisiana",
        "venice louisiana", "port-fourchon", "port fourchon", "grand-isle",
        "grand isle", "lake-charles", "lake charles", "baton-rouge",
        "baton rouge", "houma", "lafayette", "lwf", "ldwf", "wlf",
    ),
    "Texas": (
        "texas", "tx", "galveston", "corpus-christi", "corpus christi",
        "port-aransas", "port aransas", "south-padre", "south padre",
        "freeport", "houston", "brownsville", "matagorda",
        "rockport", "tpwd",
    ),
}


def extract_state(text: str) -> str:
    if not text:
        return "Gulf Council"
    s = text.lower()
    toks = set(_tokens(s))
    counts: dict[str, int] = {}
    first_pos: dict[str, int] = {}
    for state, signals in _STATE_SIGNALS.items():
        c, fp = 0, -1
        for sig in signals:
            if _matches_keyword(sig, s, toks):
                c += 1
                idx = s.find(sig.replace("-", " "))
                if idx < 0:
                    idx = s.find(sig)
                if idx >= 0 and (fp < 0 or idx < fp):
                    fp = idx
        if c:
            counts[state] = c
            first_pos[state] = fp
    if not counts:
        return "Gulf Council"
    max_c = max(counts.values())
    leaders = [st for st, c in counts.items() if c == max_c]
    if len(leaders) == 1:
        return leaders[0]
    return min(leaders, key=lambda st: first_pos[st])

## Configuration — Feeds, Targets & Transcription

The `PODCAST_FEEDS` to crawl, the download target, the pre-filter switch, and the transcription backend (HuggingFace API or local Whisper).

In [ ]:

PODCAST_FEEDS: list[dict] = [
   # ── Government / management bodies ──────────────────────────────────
    {"slug": "noaa-dive", "url": "https://www.fisheries.noaa.gov/podcast-feed",
                          "name": "Dive In with NOAA Fisheries"},

    # ── Gulf-state-specific captain shows ───────────────────────────────
    {"slug": "ccw",   "url": "https://rss2.flightcast.com/c82y8ukkqjzryba63rjlc3e6.xml",
                      "name": "Captains For Clean Water Podcast"},
    {"slug": "ss",    "url": "https://rss.libsyn.com/shows/72217/destinations/309195.xml",
                      "name": "Salt Strong Fishing"},
    {"slug": "mh",    "url": "https://feeds.megaphone.fm/MILH3045959453",
                      "name": "Mill House Podcast"},
    {"slug": "bite",  "url": "https://feed.podbean.com/highlytouted/feed.xml",
                      "name": "Bite Me — A Texas Saltwater Fishing Podcast"},
    {"slug": "lgo",   "url": "https://rss.amperwave.net/v2/feed/audacy/8db07ba6d5ffc6044c3a19fc88e80b75",
                      "name": "Louisiana Great Outdoors with Don Dubuc"},
    {"slug": "untx",  "url": "https://anchor.fm/s/e12d5d48/podcast/rss",
                      "name": "Untamed: Exploring the Texas Coast"},
    {"slug": "ec",    "url": "https://feed.podbean.com/etcurrent/feed.xml",
                      "name": "Eastern Current Saltwater Inshore Fishing"},
    {"slug": "nfo",   "url": "https://anchor.fm/s/120e0c88/podcast/rss",
                      "name": "North Florida Fishing and Outdoors"},
    {"slug": "tgcfs", "url": "https://media.rss.com/live-bait-dying-in-texas-heat-5-bait-buckets-compared/feed.xml",
                      "name": "Texas Gulf Coast Fishing Show"},
    {"slug": "gcfr",  "url": "https://anchor.fm/s/11272b740/podcast/rss",
                      "name": "Gulf Coast Fishing Report (Cactus Jack)"},
    {"slug": "tcl",   "url": "https://feeds.megaphone.fm/WPCM7699215776",
                      "name": "The Captain's Log Radio"},
    {"slug": "ysg",   "url": "https://feeds.megaphone.fm/WPCM9345573044",
                      "name": "Your Saltwater Guide Fishing Podcast"},
    {"slug": "sb",    "url": "https://rss.buzzsprout.com/775670.rss",
                      "name": "SeaBros Fishing Podcast"},

    # ── Conservation / advocacy podcasts ────────────────────────────────
    {"slug": "anchored", "url": "https://feeds.megaphone.fm/WPCM8100104444",
                         "name": "Anchored with April Vokey"},
    {"slug": "fonted",   "url": "https://anchor.fm/s/3a46110/podcast/rss",
                         "name": "Fishonted - Ted Johnson"},
    {"slug": "hooked",   "url": "https://rss.buzzsprout.com/2039085.rss",
                         "name": "Hooked the Podcast"},

    # ── Discovered via the iTunes Podcast Search API and relevance-scored

    {"slug": "pf-alabama-saltwater-fis", "url": "https://alabamasaltwaterfishingreport.libsyn.com/rss",
     "name": "Alabama Saltwater Fishing Report"},
    {"slug": "pf-brown-water-banter", "url": "https://feed.podbean.com/cmorejscott/feed.xml",
     "name": "Brown Water Banter"},
    {"slug": "pf-east-pass-podcast", "url": "https://anchor.fm/s/b4150fe0/podcast/rss",
     "name": "East Pass Podcast"},
    {"slug": "pf-the-sportsmen-s", "url": "https://feeds.megaphone.fm/SROST1319170291",
     "name": "The Sportsmen's Voice | Hunting, Fishing and Conservation Ad"},
    {"slug": "pf-islamorada-florida-fi", "url": "https://feeds.megaphone.fm/NPTNI2523806564",
     "name": "Islamorada, Florida Fishing Report Today"},
    {"slug": "pf-gulf-of-mexico", "url": "https://feeds.megaphone.fm/NPTNI8967032309",
     "name": "Gulf of Mexico, Louisiana Fishing Report Today"},
    {"slug": "pf-gulf-of-mexico2", "url": "https://feeds.megaphone.fm/NPTNI8783484305",
     "name": "Gulf of Mexico, Texas Fishing Report Today"},
    {"slug": "pf-florida-keys-fishing", "url": "https://feeds.megaphone.fm/NPTNI5125048640",
     "name": "Florida Keys Fishing Report Today"},
    {"slug": "pf-florida-keys-miami", "url": "https://feeds.megaphone.fm/NPTNI1246140443",
     "name": "Florida Keys, Miami Fishing Report Today"},
    {"slug": "pf-anglers-journal-podca", "url": "https://feeds.megaphone.fm/anglersjournalpodcast",
     "name": "Anglers Journal Podcast"},
    {"slug": "pf-game-fish", "url": "https://feeds.megaphone.fm/NPTNI5699031578",
     "name": "Game Fish"},
    {"slug": "pf-northwest-florida-fis", "url": "https://northwestfloridafishingreport.libsyn.com/rss",
     "name": "Northwest Florida Fishing Report"},
    {"slug": "pf-fishcasting-the-fishi", "url": "https://rss.buzzsprout.com/1273418.rss",
     "name": "Fishcasting the Fishing Podcast"},
    {"slug": "pf-backstory-podcast", "url": "https://feeds.soundcloud.com/users/soundcloud:users:696620853/sounds.rss",
     "name": "Backstory Podcast"},
    {"slug": "pf-srq-talk-show", "url": "https://anchor.fm/s/10e547e0/podcast/rss",
     "name": "SRQ TALK SHOW"},
    {"slug": "pf-captain-pauly-s", "url": "https://feed.podbean.com/tunatowntalks/feed.xml",
     "name": "Captain Pauly’s Podcast with Captain Paul Miller"},
    {"slug": "pf-tom-rowland-podcast", "url": "https://feeds.megaphone.fm/WPCM3342729323",
     "name": "Tom Rowland Podcast"},
    {"slug": "pf-finding-demo-surf", "url": "https://feeds.transistor.fm/finding-demo-surf-fishing",
     "name": "Finding Demo Surf Fishing"},
    {"slug": "pf-the-woman-angler", "url": "https://anchor.fm/s/eee543c4/podcast/rss",
     "name": "The Woman Angler & Adventurer"},
    {"slug": "pf-fly-fishing-daily", "url": "https://feeds.megaphone.fm/NPTNI2808851257",
     "name": "Fly Fishing Daily"},
    {"slug": "pf-florida-fishing-podca", "url": "https://feeds.soundcloud.com/users/soundcloud:users:251737620/sounds.rss",
     "name": "Florida Fishing Podcast"},
    {"slug": "pf-florida-sportsman-act", "url": "https://rss.libsyn.com/shows/199286/destinations/1417145.xml",
     "name": "Florida Sportsman Action Spotter Podcast"},
    {"slug": "pf-supertalk-outdoors-wi", "url": "https://rss.amperwave.net/v2/feed/telesouth/supertalk_outdoors___apple",
     "name": "SuperTalk Outdoors with Ricky Mathews"},
    {"slug": "pf-new-orleans-gulf", "url": "https://feeds.megaphone.fm/NPTNI5563490705",
     "name": "New Orleans Gulf of Mexico Fishing Report Today"},
    {"slug": "pf-mississippi-outdoors-", "url": "https://feeds.acast.com/public/shows/658d8d27a5c2ed0018fb9634",
     "name": "Mississippi Outdoors Podcast"},
    {"slug": "pf-good-karma-sportfishi", "url": "https://goodkarmasportfishing.libsyn.com/rss",
     "name": "Good Karma Sportfishing"},
    {"slug": "pf-the-fishing-report", "url": "https://www.omnycontent.com/d/playlist/a858b0a5-e5e6-4a14-9717-a70b010facc1/35969530-5c53-4a74-8eef-a7cf00ecd581/51e03319-5a4a-416f-a0c5-a893011bf71b/podcast.rss",
     "name": "The Fishing Report with Tina Harbuck"},
    {"slug": "pf-ripple-fishing-report", "url": "https://feeds.simplecast.com/kQROzgch",
     "name": "Ripple Fishing Report"},
    {"slug": "pf-mobile-tensaw-delta", "url": "https://feeds.redcircle.com/9c98d3d7-dbe1-4621-8594-a6f09ed3d211",
     "name": "Mobile-Tensaw Delta Fishing Report"},
    {"slug": "pf-the-marea-fishing", "url": "https://anchor.fm/s/3036c60/podcast/rss",
     "name": "The Marea Fishing PODCAST"},
    {"slug": "pf-coastal-advocacy-adve", "url": "https://feeds.soundcloud.com/users/soundcloud:users:265272006/sounds.rss",
     "name": "Coastal Advocacy Adventures Podcast"},
    {"slug": "pf-last-stop-waterfowl", "url": "https://anchor.fm/s/ef50100/podcast/rss",
     "name": "Last Stop Waterfowl Outdoors Podcast"},
    {"slug": "pf-impact-outdoors-podca", "url": "https://feeds.megaphone.fm/WPCM2819631295",
     "name": "Impact Outdoors Podcast"},
    {"slug": "pf-the-gulf-coast", "url": "https://rss.buzzsprout.com/2115059.rss",
     "name": "The Gulf Coast Food Show"},
    {"slug": "pf-the-outdoors-show", "url": "https://rss.amperwave.net/v2/feed/audacy/houston_kiltam-the_outdoors_show",
     "name": "The Outdoors Show"},
    {"slug": "pf-buckmasters-outdoors-", "url": "https://feeds.simplecast.com/L_u4qY53",
     "name": "Buckmasters Outdoors Podcast"},
    {"slug": "pf-outside-stuff-podcast", "url": "https://anchor.fm/s/370e26bc/podcast/rss",
     "name": "Outside Stuff Podcast: Florida Hunting & Fishing"},
    {"slug": "pf-louisiana-delta-fishi", "url": "https://feeds.redcircle.com/13237405-bb94-444b-a3d1-7c00d273b693",
     "name": "Louisiana Delta Fishing Report"},
    {"slug": "pf-buffalo-roamer-outdoo", "url": "https://www.spreaker.com/show/4764707/episodes/feed",
     "name": "Buffalo Roamer Outdoors"},
    {"slug": "pf-moore-outdoors-with", "url": "https://www.spreaker.com/show/2574817/episodes/feed",
     "name": "Moore Outdoors With Chester Moore"},
    {"slug": "pf-the-redfish-network", "url": "https://paddlersplaybook.podomatic.com/rss2.xml",
     "name": "The Redfish Network : The Paddlers Playbook - A Kayak fishin"},
    {"slug": "pf-catch-you-outdoors", "url": "https://anchor.fm/s/e35b4968/podcast/rss",
     "name": "Catch You Outdoors with Captain Rob Modys"},
    {"slug": "pf-fisherman-s-post", "url": "https://podcasts.helloaudio.fm/podcast/8fdcddf4-2f79-42d6-aba8-d5367508f8e5/dYuf09OLme",
     "name": "Fisherman's Post Saltwater Podcast Series"},
    {"slug": "pf-captains-collective-f", "url": "https://feeds.megaphone.fm/WPCM9140781677",
     "name": "Captains Collective Fishing Podcast"},
    {"slug": "pf-under-pressure-outdoo", "url": "https://feeds.captivate.fm/under-pressure-outdoors/",
     "name": "Under Pressure Outdoors Podcast"},
    {"slug": "pf-saltwater-edge-podcas", "url": "https://rss.libsyn.com/shows/234413/destinations/1735997.xml",
     "name": "Saltwater Edge Podcast - Sharing Our Passion For Saltwater F"},
    {"slug": "pf-lone-star-outdoor", "url": "https://lonestaroutdoorshow.com/feed/podcast/",
     "name": "Lone Star Outdoor Show"},
    {"slug": "pf-salty-a-saltwater", "url": "https://rss.buzzsprout.com/1911209.rss",
     "name": "Salty | a saltwater fly fishing podcast"},
    {"slug": "pf-the-outdoors-show2", "url": "https://www.outdoorsshow.com/feed/podcast/",
     "name": "The Outdoors Show"},
    {"slug": "pf-one-hell-of", "url": "https://rss.buzzsprout.com/1967241.rss",
     "name": "One Hell Of A Life Outdoor Podcast"},
    {"slug": "pf-state-of-sportfishing", "url": "https://rss.buzzsprout.com/1815814.rss",
     "name": "State of Sportfishing - Billfish Group"},
    {"slug": "pf-kayak-fishing-with", "url": "https://feeds.soundcloud.com/users/soundcloud:users:435803532/sounds.rss",
     "name": "Kayak Fishing with Jim Sammons"},
    {"slug": "pf-tall-tails-fishing", "url": "https://rss.buzzsprout.com/2450969.rss",
     "name": "Tall Tails Fishing Podcast"},
    {"slug": "pf-fish-nerds-fishing", "url": "https://rss.libsyn.com/shows/79086/destinations/360762.xml",
     "name": "Fish Nerds Fishing Podcast"},
    {"slug": "pf-the-orvis-fly", "url": "https://orvisffguide.libsyn.com/rss",
     "name": "The Orvis Fly-Fishing Podcast"},
    {"slug": "pf-dockside-with-captain", "url": "https://kacey-lanier-ddmq.squarespace.com/podcast?format=rss",
     "name": "Dockside with Captain Kevin"},
    {"slug": "pf-off-the-water", "url": "https://feeds.soundcloud.com/users/soundcloud:users:585010488/sounds.rss",
     "name": "Off The Water - Kayak Fishing Podcast"},
    {"slug": "pf-deep-water-legends", "url": "https://media.rss.com/deepwaterlegends-goin-deep-offshore-fishing-podcast/feed.xml",
     "name": "Deep Water Legends: Goin’ Deep – Offshore Fishing Podcast"},
    {"slug": "pf-serious-angler-bass", "url": "https://rss.buzzsprout.com/2109808.rss",
     "name": "Serious Angler Bass Fishing Podcast"},
    {"slug": "pf-sportsmen-s-empire", "url": "https://feeds.redcircle.com/7bba5a6f-2afb-43a9-820b-6b97e8a1f04e",
     "name": "Sportsmen's Empire - Whitetail Hunting"},
    {"slug": "pf-the-alex-rudd", "url": "https://anchor.fm/s/fd5c9a74/podcast/rss",
     "name": "The Alex Rudd Fishing Podcast"},
    {"slug": "pf-john-graves-kayak", "url": "https://anchor.fm/s/bd9e8dc/podcast/rss",
     "name": "John Graves Kayak Fishing"},
    {"slug": "pf-fly-fishing-insider", "url": "https://rss.libsyn.com/shows/163634/destinations/1082348.xml",
     "name": "Fly Fishing Insider Podcast"},
    {"slug": "pf-trilogy-outdoors", "url": "https://www.spreaker.com/show/5441492/episodes/feed",
     "name": "Trilogy Outdoors"},
    {"slug": "pf-backcountry-hunters-a", "url": "https://rss.libsyn.com/shows/102260/destinations/542457.xml",
     "name": "Backcountry Hunters & Anglers Podcast & Blast with Hal Herri"},
    {"slug": "pf-misguided-fishing", "url": "https://rss.buzzsprout.com/1153874.rss",
     "name": "Misguided Fishing"},
    {"slug": "pf-the-fisheries-podcast", "url": "https://feed.podbean.com/fisheriespodcast/feed.xml",
     "name": "The Fisheries Podcast"},
    {"slug": "pf-all-the-gear", "url": "https://rss.buzzsprout.com/778907.rss",
     "name": "All The Gear But No Idea - The South Australian Fishing Podc"},
    {"slug": "pf-capt-marty-s", "url": "https://rss.buzzsprout.com/2312605.rss",
     "name": "Capt. Marty's Outer Banks Fishing Report & Stories"},
    {"slug": "pf-retirecoast", "url": "https://rss.buzzsprout.com/1948879.rss",
     "name": "RetireCoast"},
    {"slug": "pf-the-barbless-co", "url": "https://anchor.fm/s/57a93470/podcast/rss",
     "name": "The Barbless.co Fly Fishing Podcast with Hogan Brown"},
    {"slug": "pf-backwater-hustle-the", "url": "http://rss.castbox.fm/everest/6e914a02909949dda6ca2d215bece890.xml",
     "name": "Backwater Hustle, The Fishing Podcast"},
    {"slug": "pf-blind-hog-adventures", "url": "https://feeds.podcastle.ai/6dcf87eb-7918-411e-81e5-5c1b1c9352c3.rss",
     "name": "Blind Hog Adventures Inshore Saltwater Fishing Podcast"},
    {"slug": "pf-blind-hog-adventures2", "url": "https://feeds.async.com/6dcf87eb-7918-411e-81e5-5c1b1c9352c3.rss",
     "name": "Blind Hog Adventures Inshore Saltwater Fishing Podcast"},
    {"slug": "pf-coffee-with-the", "url": "https://anchor.fm/s/a563d1c/podcast/rss",
     "name": "Coffee with the Captain"},
    {"slug": "pf-hunt-fish-travel", "url": "https://rss.libsyn.com/shows/37330/destinations/99828.xml",
     "name": "Hunt Fish Travel Podcast with Carrie Z, a podcast about hunt"},
    {"slug": "pf-the-reel-action", "url": "https://rss.libsyn.com/shows/257009/destinations/1946633.xml",
     "name": "The Reel Action Fishing Podcast (with Guesty and the Ferret)"},
    {"slug": "pf-aptitude-outdoors-pod", "url": "https://rss.libsyn.com/shows/240338/destinations/1794200.xml",
     "name": "Aptitude Outdoors Podcast"},
    {"slug": "pf-australian-lure-fishi", "url": "https://rss.libsyn.com/shows/139244/destinations/870077.xml",
     "name": "Australian Lure Fishing"},
    {"slug": "pf-the-sustainable-angle", "url": "https://rss.buzzsprout.com/2464854.rss",
     "name": "The Sustainable Angler"},
    {"slug": "pf-the-nomadic-outdoorsm", "url": "https://feeds.simplecast.com/Kuqm24oM",
     "name": "The Nomadic Outdoorsman"},
    {"slug": "pf-fly-fishing-consultan", "url": "https://feeds.megaphone.fm/WPCM9859557776",
     "name": "Fly Fishing Consultant Podcast"},
    {"slug": "pf-the-anglers-fishing", "url": "https://www.omnycontent.com/d/playlist/820f09cf-2ace-4180-a92d-aa4c0008f5fb/bbedba06-d200-471c-a530-afcc00579ccf/fa62a59b-2dd7-44f6-85da-afcc005804e0/podcast.rss",
     "name": "The Anglers Fishing Podcast"},
    {"slug": "pf-all-things-outdoors", "url": "https://rss.libsyn.com/shows/616270/destinations/5375515.xml",
     "name": "All Things Outdoors"},
    {"slug": "pf-fishtory-podcast", "url": "https://anchor.fm/s/4bc75d80/podcast/rss",
     "name": "Fishtory Podcast"},
]

# Maximum episodes to consider per feed (most recent first).
MAX_EPISODES_PER_FEED = 150

# Default total target (passing-filter episodes saved across the whole run).
TARGET_EPISODES = 6000

## Transcription Backend

Selects how episode audio becomes text: the HuggingFace hosted Whisper API when `HF_API_KEY` is set, otherwise a local faster-whisper model. Also sets the Gulf pre-filter switch and the download worker count.

In [ ]:

TRANSCRIPT_BACKEND = "hf"
# hugginface api key
# Set HF_API_KEY via environment or .env (never hardcode tokens)

HF_API_KEY = (
    os.environ.get("HF_API_KEY")
    or os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or "YOUR_HF_API_KEY_HERE"  # Replace with your actual Hugging Face API key
)

if HF_API_KEY:
    os.environ["HF_TOKEN"] = HF_API_KEY
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_API_KEY
HF_MODEL   = "openai/whisper-large-v3"


WHISPER_MODEL = "tiny"


PREFILTER_REQUIRES_GULF = True

# Parallelism — keep low because audio downloads + transcription are heavy.
DOWNLOAD_WORKERS = 8

## Paths & Metadata Schema

Output locations under the shared `Data_Repository` and the seven-field metadata schema written for every saved episode.

In [ ]:
ROOT         = Path.cwd().resolve()
DATA         = ROOT / "Data_Repository"
RAW_TXT_DIR  = DATA / "scraped_text" / "youtube_podcast"
METADATA_DIR = DATA / "metadata" / "youtube_podcast"
RECORDS_DIR  = DATA / "records"
STATE_FILE   = DATA / "podcast_state.json"

METADATA_FIELDS = (
    "Document_ID", "Source_Type", "Source_Name", "URL", "Title",
    "Published_Date", "Location",
)

## Dependency Check & Whisper Loader

Verifies feedparser / requests are present and lazily loads the local Whisper model when the local backend is used.

In [ ]:
try:
    import feedparser  # type: ignore
    _FEEDPARSER = True
except ImportError:
    _FEEDPARSER = False
try:
    import requests  # type: ignore
    _REQUESTS = True
except ImportError:
    _REQUESTS = False
try:
    from faster_whisper import WhisperModel  # type: ignore
    _WHISPER = True
except ImportError:
    _WHISPER = False

_whisper_model = None  # lazy-loaded singleton


def _require_deps():
    missing = []
    if not _FEEDPARSER: missing.append("feedparser")
    if not _REQUESTS:   missing.append("requests")
    # Local Whisper is only required if HF path is unavailable.
    if not _WHISPER and not HF_API_KEY:
        missing.append("faster-whisper (or set HF_API_KEY)")
    if missing:
        sys.exit(f"Missing deps: {missing}.\n"
                 f"Run: pip install -U feedparser requests faster-whisper")


def _get_whisper():
    global _whisper_model
    if _whisper_model is None:
        # Auto-detect GPU on Colab; CPU otherwise.
        try:
            _whisper_model = WhisperModel(WHISPER_MODEL, device="auto",
                                          compute_type="auto")
        except Exception:
            _whisper_model = WhisperModel(WHISPER_MODEL, device="cpu",
                                          compute_type="int8")
    return _whisper_model

## RSS Feed Parsing

Reads each podcast RSS feed and extracts per-episode title, description, audio URL, publish date and GUID.

In [ ]:
def fetch_feed(url: str, limit: int) -> tuple[str, list[dict]]:
    """Return (feed_title, episodes). Each episode is a dict with
    title, description, audio_url, episode_url, published, guid."""
    try:
        parsed = feedparser.parse(url)
    except Exception as e:
        print(f"  [feed] {url}: {type(e).__name__}: {e}")
        return "", []
    feed_title = (parsed.get("feed", {}) or {}).get("title", "") or ""
    entries = parsed.get("entries", [])[:limit]
    episodes: list[dict] = []
    for e in entries:
        # Find the audio enclosure URL
        audio_url = ""
        for enc in e.get("enclosures", []) or []:
            t = (enc.get("type") or "").lower()
            if "audio" in t or "mpeg" in t or "mp3" in t:
                audio_url = enc.get("href") or enc.get("url") or ""
                break
            if not audio_url and enc.get("href"):
                # Fallback: any enclosure ending in audio extension
                href = enc.get("href")
                if href and href.lower().endswith((".mp3", ".m4a", ".wav",
                                                    ".ogg", ".flac")):
                    audio_url = href
                    break
        if not audio_url:
            continue
        episodes.append({
            "title":       (e.get("title") or "").strip(),
            "description": (e.get("summary") or e.get("description") or "").strip(),
            "audio_url":   audio_url,
            "episode_url": e.get("link") or "",
            "published":   e.get("published") or e.get("updated") or "",
            "guid":        e.get("id") or e.get("guid") or audio_url,
        })
    return feed_title, episodes

## Audio Download & Transcription

Downloads each episode's audio and transcribes it via the HuggingFace API or local faster-whisper.

In [ ]:
def download_audio(audio_url: str, dest: Path) -> bool:
    """Stream-download audio to `dest`. Returns True on success."""
    try:
        with requests.get(audio_url, stream=True, timeout=60,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            r.raise_for_status()
            with dest.open("wb") as f:
                for chunk in r.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        f.write(chunk)
        return dest.stat().st_size > 10_000  # >10KB sanity check
    except Exception:
        return False


_hf_first_error_logged = False


def _log_hf_error(status: int, body: str):
    global _hf_first_error_logged
    if _hf_first_error_logged:
        return
    _hf_first_error_logged = True
    print(f"  [hf-api] status={status}  body[:200]={body[:200]!r}")
    print(f"  [hf-api] further HF errors silenced; falling back to local whisper")


def _transcribe_hf(audio_path: Path) -> str:
    """Use HuggingFace Inference API (Whisper-large-v3). Returns text or ''
    on failure. Falls back to local whisper in the caller on errors."""
    if not HF_API_KEY:
        return ""
    url = f"https://api-inference.huggingface.co/models/{HF_MODEL}"
    headers = {
        "Authorization": f"Bearer {HF_API_KEY}",
        "Content-Type":  "audio/mpeg",
    }
    try:
        data = audio_path.read_bytes()
        # HF Inference API can warm-cold-start the model; allow retries.
        for attempt in range(3):
            try:
                r = requests.post(url, headers=headers, data=data, timeout=300)
            except Exception:
                time.sleep(2 ** attempt)
                continue
            if r.status_code == 503:
                # Model loading — wait + retry
                try:
                    wait = float(r.json().get("estimated_time", 5))
                except Exception:
                    wait = 5.0
                time.sleep(min(wait, 30))
                continue
            if r.status_code != 200:
                _log_hf_error(r.status_code, r.text)
                return ""
            try:
                payload = r.json()
            except Exception:
                return ""
            if isinstance(payload, dict):
                return (payload.get("text") or "").strip()
            return ""
    except Exception:
        return ""
    return ""


def _transcribe_local(audio_path: Path) -> str:
    """Run faster-whisper locally. Returns text or ''."""
    if not _WHISPER:
        return ""
    try:
        model = _get_whisper()
        segments, _info = model.transcribe(
            str(audio_path),
            beam_size=1, vad_filter=True, language="en",
        )
        return " ".join(seg.text.strip() for seg in segments).strip()
    except Exception:
        return ""


def transcribe_audio(audio_path: Path) -> str:
    """Transcribe via the configured backend, falling back as needed."""
    if TRANSCRIPT_BACKEND == "hf":
        text = _transcribe_hf(audio_path)
        if text:
            return text
        # HF failed (no key, rate limit, etc.) — try local if available
        return _transcribe_local(audio_path)
    # local-first
    text = _transcribe_local(audio_path)
    if text:
        return text
    return _transcribe_hf(audio_path)

## Output & CSV

Writes each saved episode's `.txt` and `.json`, and rebuilds the consolidated `podcasts.csv` index.

In [ ]:
def render_doc_txt(doc: dict) -> str:
    lines = [
        f"Document_ID  : {doc['Document_ID']}",
        f"Source_Type  : {doc['Source_Type']}",
        f"Source_Name  : {doc['Source_Name']}",
        f"URL          : {doc['URL']}",
        f"Title        : {doc['Title']}",
        f"Published    : {doc.get('Published_Date','')}",
        f"Location     : {doc['Location']}",
        "=" * 80,
        "",
        doc.get("_text", ""),
    ]
    return "\n".join(lines)


def write_doc(doc: dict):
    tid = doc["Document_ID"]
    (RAW_TXT_DIR / f"{tid}.txt").write_text(render_doc_txt(doc), encoding="utf-8")
    meta = {k: doc.get(k, "") for k in METADATA_FIELDS}
    (METADATA_DIR / f"{tid}.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
    )


def rebuild_csv() -> int:
    csv_path = RECORDS_DIR / "youtube_podcast.csv"
    rows: list[dict] = []
    for jf in sorted(METADATA_DIR.glob("*.json")):
        try:
            d = json.loads(jf.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(d, dict) or "Document_ID" not in d:
            continue
        rows.append({k: d.get(k, "") for k in METADATA_FIELDS})
    if rows:
        with csv_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(METADATA_FIELDS),
                               quoting=csv.QUOTE_ALL)
            w.writeheader()
            w.writerows(rows)
    return len(rows)

## Run State

Loads and saves resumable progress (seen episode GUIDs) in `podcast_state.json`.

In [ ]:
def load_state() -> dict:
    if STATE_FILE.exists():
        try:
            return json.loads(STATE_FILE.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {"seen_guids": [], "feed_stats": {}}


def save_state(s: dict):
    STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    STATE_FILE.write_text(json.dumps(s, indent=1, ensure_ascii=False),
                          encoding="utf-8")

## Run — Podcast Crawl

Orchestrates Part 2: read feeds → pre-filter by Gulf relevance **and the last-10-years window** (before downloading audio) → transcribe → strict gate → save → rebuild the CSV.

In [ ]:
def _guid_to_docid(slug: str, guid: str) -> str:
    """Stable doc id from feed slug + episode guid."""
    import hashlib
    h = hashlib.md5(guid.encode("utf-8")).hexdigest()[:12]
    return f"pod_{slug}_{h}"


def _normalize_pub_date(s: str) -> str:
    """RSS pub dates are RFC822; convert to ISO YYYY-MM-DD when possible."""
    if not s:
        return ""
    try:
        from email.utils import parsedate_to_datetime
        dt = parsedate_to_datetime(s)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return s


DATA_WINDOW_YEARS = 10

def within_data_window(date_str: str, years: int = DATA_WINDOW_YEARS) -> bool:
    """True if `date_str` (ISO YYYY-MM-DD or similar) falls within the last
    `years` years. Reads the first 4-digit year; empty / unparseable dates
    return True so we never drop media we can't date."""
    if not date_str:
        return True
    m = re.search(r"(?:19|20)\d{2}", date_str)
    if not m:
        return True
    return int(m.group(0)) >= datetime.now().year - years



def prefilter_episode(ep: dict) -> bool:
    """Cheap title+description filter to skip non-Gulf episodes BEFORE
    spending bandwidth + CPU on audio transcription."""
    # Last-N-years window: skip stale episodes BEFORE downloading audio.
    if not within_data_window(_normalize_pub_date(ep.get("published", ""))):
        return False
    blob = f"{ep['title']}\n{ep['description']}"
    if PREFILTER_REQUIRES_GULF and not gulf_hits(blob):
        return False
    return True


def process_episode(ep: dict, feed_slug: str, feed_name: str
                    ) -> tuple[Optional[dict], str]:
    """Download audio, transcribe, apply strict gate. Returns (doc, outcome)."""
    with tempfile.TemporaryDirectory() as td:
        audio_path = Path(td) / "episode.mp3"
        if not download_audio(ep["audio_url"], audio_path):
            return None, "download_failed"
        text = transcribe_audio(audio_path)
    if not text or len(text) < 400:
        return None, "transcribe_failed_or_too_short"
    blob = f"{ep['title']}\n{ep['description']}\n{text}"
    phits = policy_hits(blob)
    if not phits:
        return None, "no_policy"
    ghits = gulf_hits(blob)
    if not ghits:
        return None, "no_gulf"
    strong_p = [h for h in phits if h in _STRONG_POLICY_SET]
    strong_g = [h for h in ghits if h in _STRONG_GULF_SET]
    if not (strong_p and strong_g):
        return None, "no_strong"
    doc_id = _guid_to_docid(feed_slug, ep["guid"])
    doc = {
        "Document_ID":    doc_id,
        "Source_Type":    "podcast",
        "Source_Name":    feed_name,
        "URL":            ep["episode_url"] or ep["audio_url"],
        "Title":          ep["title"][:500],
        "Published_Date": _normalize_pub_date(ep["published"]),
        "Location":       extract_state(blob),
        "_text":          text,
    }
    return doc, "saved"


def run(target_total: int = TARGET_EPISODES,
        only_feeds: str = "",
        rebuild_csv_only: bool = False) -> int:
    _require_deps()

    for p in (DATA, RAW_TXT_DIR, METADATA_DIR, RECORDS_DIR):
        p.mkdir(parents=True, exist_ok=True)

    if rebuild_csv_only:
        n = rebuild_csv()
        print(f"Rebuilt youtube_podcast.csv with {n} rows.")
        return n

    state = load_state()
    seen: set[str] = set(state.get("seen_guids", []))
    only_set = {s.strip() for s in only_feeds.split(",") if s.strip()}

    if TRANSCRIPT_BACKEND == "hf":
        backend_label = f"HuggingFace API ({HF_MODEL})" if HF_API_KEY \
                        else "HF (NO KEY → will fall back to local)"
    else:
        backend_label = f"local faster-whisper ({WHISPER_MODEL})"
    print(f"[cfg] target_total       = {target_total}")
    print(f"[cfg] feeds              = {len(PODCAST_FEEDS)}")
    print(f"[cfg] transcript backend = {backend_label}")
    print(f"[cfg] download workers   = {DOWNLOAD_WORKERS}")
    print(f"[cfg] prefilter gulf-only= {PREFILTER_REQUIRES_GULF}")
    print(f"[cfg] already-seen GUIDs = {len(seen)}")
    print()

    # ── Phase 1: enumerate episodes across all feeds, apply prefilter.
    candidates: list[tuple[dict, str, str]] = []
    for f in PODCAST_FEEDS:
        if only_set and f["slug"] not in only_set:
            continue
        feed_title, episodes = fetch_feed(f["url"], MAX_EPISODES_PER_FEED)
        name = feed_title or f["name"]
        new = [e for e in episodes if e["guid"] not in seen]
        passed = [e for e in new if prefilter_episode(e)]
        print(f"[feed] {f['slug']:<10} {name[:42]:<42}  "
              f"episodes={len(episodes)} new={len(new)} after-prefilter={len(passed)}")
        for ep in passed:
            candidates.append((ep, f["slug"], name))
        state["feed_stats"][f["slug"]] = {
            "name": name, "total_listed": len(episodes), "new": len(new),
            "after_prefilter": len(passed),
            "at": datetime.now(timezone.utc).isoformat(),
        }
    print()
    print(f"[queue] {len(candidates)} episodes to download+transcribe")

    # ── Phase 2: download + transcribe + filter + save.
    saved = 0
    examined = 0
    outcomes: dict[str, int] = {}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {
            pool.submit(process_episode, ep, slug, name): ep["guid"]
            for ep, slug, name in candidates
        }
        for fut in as_completed(futures):
            guid = futures[fut]
            seen.add(guid)
            examined += 1
            try:
                doc, outcome = fut.result()
            except Exception:
                outcomes["worker_error"] = outcomes.get("worker_error", 0) + 1
                continue
            outcomes[outcome] = outcomes.get(outcome, 0) + 1
            if doc is not None:
                write_doc(doc)
                saved += 1
                safe_title = doc["Title"][:65].encode("ascii", "replace").decode("ascii")
                print(f"  [{saved:>4}/{target_total}] {doc['Document_ID']:<28} "
                      f"loc={doc['Location']:<13}  {safe_title}")
            if saved >= target_total:
                break

    state["seen_guids"] = sorted(seen)
    save_state(state)
    total_rows = rebuild_csv()
    elapsed = time.time() - t0
    print()
    print(f"=== youtube_podcast.csv rebuilt: {total_rows} total rows ===")
    print(f"=== this run: examined {examined} episodes, saved {saved} in {elapsed:.1f}s ===")
    print("=== outcome breakdown ===")
    order = ["saved", "no_strong", "no_gulf", "no_policy",
             "transcribe_failed_or_too_short", "download_failed",
             "worker_error"]
    for k in order:
        if k in outcomes:
            print(f"     {outcomes[k]:>5}  {k}")
    for k, v in outcomes.items():
        if k not in order:
            print(f"     {v:>5}  {k}")
    return total_rows


def main():
    ap = argparse.ArgumentParser(description="Audio podcast Gulf-Council / FEI scraper")
    ap.add_argument("-n", "--target", type=int, default=TARGET_EPISODES,
                    help=f"target number of saved episodes (default {TARGET_EPISODES})")
    ap.add_argument("--only", type=str, default="",
                    help="comma-separated feed slugs (e.g. ccw,mh,bite)")
    ap.add_argument("--rebuild-csv-only", action="store_true",
                    help="skip fetch, rebuild podcasts.csv from existing JSONs")
    args, _unknown = ap.parse_known_args()
    run(target_total=args.target,
        only_feeds=args.only,
        rebuild_csv_only=args.rebuild_csv_only)


if __name__ == "__main__":
    main()